In [ ]:
from astropy.io import fits
from astropy.coordinates import SkyCoord
from astropy.coordinates import ICRS, Galactic, FK4, FK5
import numpy as np
import matplotlib.pyplot as plt
from astropy.coordinates import angular_separation
from astropy.wcs import WCS
import gc
from scipy.stats import pearsonr

## Multi-panel plot function

In [ ]:
def make_landecker_mapPI(data,hdrs,vmax=[1,1,1,1],cmap='viridis',
                         llim = [192,52], blim = [-7,10],filename='test',
                         *args,**kwargs):
    
    c = SkyCoord(llim, blim, frame=Galactic, unit="deg")
    fs = 16
    
    # Set up axes:
    fig = plt.figure(figsize=(26,10))   
    plt.subplots_adjust(hspace=0.1,wspace=0.12,left=0.04, right=0.98, top=0.95, bottom=0.1)  
    ax1  = fig.add_subplot(231, projection=WCS(hdrs[0]).celestial)
    ax2  = fig.add_subplot(232, projection=WCS(hdrs[1]).celestial)
    ax3  = fig.add_subplot(234, projection=WCS(hdrs[2]).celestial)
    ax4  = fig.add_subplot(235, projection=WCS(hdrs[3]).celestial)
    ax5  = fig.add_subplot(133) 
    axes = [ax1,ax2,ax3,ax4,ax5]
    
    # The maps:
    ims   = []
    for i in range(0,4):
        ims.append(axes[i].imshow(data[i],origin='lower',vmin=0, vmax=vmax[i],cmap=cmap))
        axes[i].set_xlim(WCS(hdrs[i]).world_to_pixel(c)[0])
        axes[i].set_ylim(WCS(hdrs[i]).world_to_pixel(c)[1])
    axes[0].set_title(r'DRAO 26-m PI (K)', fontsize=fs)
    axes[1].set_title(r'CGPS PI (K)', fontsize=fs)
    axes[2].set_title(r'GMIMS PI (K)', fontsize=fs)
    axes[3].set_title(r'ST+GMIMS PI (K)', fontsize=fs)
    
    # The colorbars
    cbar1 = fig.colorbar(ims[2],ax=axes[2],orientation='horizontal',
                         fraction=0.1,pad=0.1,aspect=20)
    cbar2 = fig.colorbar(ims[3],ax=axes[3],orientation='horizontal',
                         fraction=0.1,pad=0.1,aspect=20)
    for cbar in [cbar1,cbar2]:
        cbar.set_ticks([0,0.1,0.2,0.3,0.4,0.5])
        cbar.set_label(r'PI (K)', fontsize=fs)
        cbar.ax.tick_params(axis='y', which='both', width=2, length=6)
        cbar.ax.tick_params(labelsize=fs)
        cbar.outline.set_linewidth(2)
    
    for ax in [ax1,ax2,ax3,ax4,ax5]:
        ax.tick_params(axis='both', labelsize=fs)
        ax.set_ylabel('  ',fontsize=fs)
        ax.set_xlabel('  ',fontsize=fs)
        ax.tick_params(axis='both', which='both', width=2, length=6)
        for spine in ax.spines.values():
            spine.set_visible(True)
            spine.set_linewidth(2)
    ax1.set_xlabel('Galactic Longitude',fontsize=fs)
    ax2.set_xlabel('Galactic Longitude',fontsize=fs)
    ax3.set_ylabel('Galactic Latitude',fontsize=fs)
    ax1.set_ylabel('Galactic Latitude',fontsize=fs)
    
    # The 2D histogram:
    i1 = int(np.round(WCS(hdrs[1]).world_to_pixel(c)[0][0],0))
    i2 = int(np.round(WCS(hdrs[1]).world_to_pixel(c)[0][1],0))
    j1 = int(np.round(WCS(hdrs[1]).world_to_pixel(c)[1][0],0))
    j2 = int(np.round(WCS(hdrs[1]).world_to_pixel(c)[1][1],0))
    print(i1,i2,j1,j2)
    data1_sub = data[1][j1:j2,i1:i2]
    data2_sub = data[3][j1:j2,i1:i2] 
    ax5.hist2d(data1_sub[np.isfinite(data1_sub)].flatten(),
               data2_sub[np.isfinite(data1_sub)].flatten(),
               range=([[0,1],[0,1]]),bins=(500,500), 
               cmap='cubehelix',vmin=0,vmax=500);
    ax5.set_aspect('equal')
    ax5.plot([0,1],[0,1],linewidth=2,color='w')
    ax5.set_xlim(0,0.8)
    ax5.set_ylim(0,0.8)
    ax5.set_xlabel('CGPS PI (K)',fontsize=fs)
    ax5.set_ylabel('ST + GMIMS PI (K)',fontsize=fs)
    
    #plt.savefig('/home/aordog/CGPS_GMIMS_PLOTS/'+filename+'.jpg')

    return

In [ ]:
def TT_plots_2006_2010(data,hdrs,vmax=[1,1,1,1],cmap='viridis',
                         llim = [192,52], blim = [-7,10],filename='test',
                         *args,**kwargs):
    
    c = SkyCoord(llim, blim, frame=Galactic, unit="deg")
    fs = 16
    
    # Set up axes:
    #fig = plt.figure(figsize=(26,10))
    fig = plt.figure(figsize=(10,10))   
    ax  = fig.add_subplot(111) 

    ax.tick_params(axis='both', labelsize=fs)
    ax.set_ylabel('  ',fontsize=fs)
    ax.set_xlabel('  ',fontsize=fs)
    ax.tick_params(axis='both', which='both', width=2, length=6)
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(2)

    # The 2D histogram:
    i1_1 = int(np.floor(WCS(hdrs[0]).world_to_pixel(c)[0][0]))
    i2_1 = int(np.floor(WCS(hdrs[0]).world_to_pixel(c)[0][1]))
    j1_1 = int(np.floor(WCS(hdrs[0]).world_to_pixel(c)[1][0]))
    j2_1 = int(np.floor(WCS(hdrs[0]).world_to_pixel(c)[1][1]))
    print(i1_1,i2_1,j1_1,j2_1)
    data1_sub = data[0][j1_1:j2_1,i1_1:i2_1]/1e3

    i1_2 = int(np.floor(WCS(hdrs[2]).world_to_pixel(c)[0][0]))
    i2_2 = int(np.floor(WCS(hdrs[2]).world_to_pixel(c)[0][1]))
    j1_2 = int(np.floor(WCS(hdrs[2]).world_to_pixel(c)[1][0]))
    j2_2 = int(np.floor(WCS(hdrs[2]).world_to_pixel(c)[1][1]))
    print(i1_2,i2_2,j1_2,j2_2)
    data2_sub = data[2][j1_2:j2_2,i1_2:i2_2]
    
    #ax.hist2d(data1_sub[np.isfinite(data1_sub)].flatten(),
    #           data2_sub[np.isfinite(data1_sub)].flatten(),
    #           range=([[0,1],[0,1]]),bins=(100,100), 
    #           cmap='cubehelix',vmin=0,vmax=50);
    ax.scatter(data1_sub[np.isfinite(data1_sub)].flatten(),
               data2_sub[np.isfinite(data1_sub)].flatten(),s=5)
    ax.set_aspect('equal')
    ax.plot([0,1],[0,1],linewidth=2,color='k')
    ax.set_xlim(0,0.8)
    ax.set_ylim(0,0.8)
    ax.set_xlabel('DRAO 2006 (K)',fontsize=fs)
    ax.set_ylabel('GMIMS (K)',fontsize=fs)
    
    #plt.savefig('/home/aordog/CGPS_GMIMS_PLOTS/'+filename+'.jpg')

    return

## Read in the data

In [ ]:
# (1) Wolleben et al 2006 26-m data (12 MHz bandwidth)
hdu_26Q = fits.open('/srv/data/cgps/drao26m_2006/q_gal.fit')
Q_26 = hdu_26Q[0].data
hdu_26U = fits.open('/srv/data/cgps/drao26m_2006/u_gal.fit')
U_26 = hdu_26U[0].data
PI_26 = np.sqrt(Q_26**2+U_26**2)
hdr_26 = hdu_26U[0].header

# (2) Landecker et al 2010 CGPS ST+Galt+Effelsberg data (from CADC)
#hdu_PI_cadc = fits.open('/srv/aordog/cgps_gmims_data/cgps_cadc_PI.fits')
hdu_PI_cadc = fits.open('/srv/data/cgps-gmims/cgps_cadc/cgps_cadc_PI.fits')
PI_cadc = hdu_PI_cadc[0].data
hdr_cadc = hdu_PI_cadc[0].header

# (3) Wolleben et al 2021 GMIMS-HBN (interpolated=35 MHz, OG=12 MHz)
interpolated = False
if interpolated:
    #hdu_G = fits.open('/srv/aordog/cgps_gmims_data/PI_G_regrd_avg_PI.fits')
    hdu_G = fits.open('/srv/data/cgps-gmims/raw_derived/ PI_G.fits')
    PI_G = hdu_G[0].data
    hdr_G = hdu_G[0].header
else:
    hdu_G = fits.open('/srv/data/gmims/gmims-hbn/GMIMS-HBN_v1_gal_car_freq_IQU.fits')
    Q_G = np.nanmean(hdu_G[0].data[1][116:127],axis=0)
    U_G = np.nanmean(hdu_G[0].data[2][116:127],axis=0)
    PI_G = np.sqrt(Q_G**2+U_G**2)
    hdr_G = hdu_G[0].header
    hdr_G['NAXIS'] = 2
    hdr_G['WCSAXES'] = 2
    del hdr_G['NAXIS3']
    del hdr_G['NAXIS4']
    del hdr_G['CRPIX3']
    del hdr_G['CRPIX4']
    del hdr_G['CDELT3']
    del hdr_G['CDELT4']
    del hdr_G['CRVAL3']
    del hdr_G['CRVAL4']
    del hdr_G['CTYPE3']
    del hdr_G['CTYPE4']
    del hdr_G['CUNIT3']

# (4) Ordog et al 2023 ST+GMIMS-HBN (35 MHz bandwidth)
#hdu_PI_CG = fits.open('/srv/aordog/cgps_gmims_data/PI_CG.fits')
hdu_PI_CG = fits.open('/srv/data/cgps-gmims/raw_derived/PI_CG.fits')
PI_CG     = hdu_PI_CG[0].data
hdr_CG = hdu_PI_CG[0].header


## Make maps and comparisons

In [ ]:
l1 = 120
l2 = 140

make_landecker_mapPI([PI_26, PI_cadc, PI_G, PI_CG],
                     [hdr_26,hdr_cadc,hdr_G,hdr_CG],
                     vmax=[600,0.6,0.6,0.6],cmap='gist_heat_r',
                     llim = [l2,l1], blim = [-3,5],
                     filename = 'CGPS_PI_compare_'+str(l1)+'_'+str(l2))

In [ ]:
c = SkyCoord([180,60], [-30,30], frame=Galactic, unit="deg")
fs = 14

# Set up axes:
fig = plt.figure(figsize=(26,20))   
plt.subplots_adjust(hspace=0.1,wspace=0.12,left=0.04, right=0.98, top=0.95, bottom=0.1)  
ax1  = fig.add_subplot(211, projection=WCS(hdr_26).celestial)
ax2  = fig.add_subplot(212, projection=WCS(hdr_G).celestial)
axes = [ax1,ax2]

# The maps:
ims   = []
ims.append(ax1.imshow(PI_26/1e3,origin='lower',vmin=0, vmax=0.5,cmap='afmhot'))
ims.append(ax2.imshow(PI_G,origin='lower',vmin=0, vmax=0.5,cmap='afmhot'))

ax1.set_xlim(WCS(hdr_26).world_to_pixel(c)[0])
ax1.set_ylim(WCS(hdr_26).world_to_pixel(c)[1])

ax2.set_xlim(WCS(hdr_G).world_to_pixel(c)[0])
ax2.set_ylim(WCS(hdr_G).world_to_pixel(c)[1])

# The colorbars
cbar1 = fig.colorbar(ims[0],ax=ax1,orientation='horizontal',
                     fraction=0.02,pad=0.1,aspect=40)
cbar2 = fig.colorbar(ims[1],ax=ax2,orientation='horizontal',
                     fraction=0.02,pad=0.1,aspect=40)
for cbar in [cbar1,cbar2]:
    cbar.set_ticks([0,0.1,0.2,0.3,0.4,0.5])
    cbar.set_label(r'PI (K)', fontsize=fs)
    cbar.ax.tick_params(axis='y', which='both', width=0.1, length=4)
    cbar.ax.tick_params(labelsize=fs)
    cbar.outline.set_linewidth(2)

for ax in [ax1,ax2]:
    ax.tick_params(axis='both', labelsize=fs)
    ax.set_ylabel('  ',fontsize=fs)
    ax.set_xlabel('  ',fontsize=fs)
    ax.tick_params(axis='both', which='both', width=2, length=6)
    ax.set_xlabel('Galactic Longitude',fontsize=fs)
    ax.set_ylabel('Galactic Latitude',fontsize=fs)
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(2)

In [ ]:
l1 = 120
l2 = 140

TT_plots_2006_2010([PI_26, PI_cadc, PI_G, PI_CG],
                     [hdr_26,hdr_cadc,hdr_G,hdr_CG],
                     vmax=[600,0.6,0.6,0.6],cmap='gist_heat_r',
                     llim = [l2,l1], blim = [-3,5],
                     filename = 'CGPS_PI_compare_'+str(l1)+'_'+str(l2))

In [ ]:
fig = plt.figure(figsize=(6,6))

ax1 = fig.add_subplot(111)
ax1.hist2d(PI_CG[np.isfinite(PI_cadc)].flatten(),PI_cadc[np.isfinite(PI_cadc)].flatten(),
           range=([[0,1],[0,1]]),bins=(500,500), cmap='cubehelix',vmin=0,vmax=2000);
ax1.plot([0,1],[0,1],linewidth=2,color='w')
ax1.set_xlim(0,0.8)
ax1.set_ylim(0,0.8)
#plt.savefig('/home/aordog/CGPS_GMIMS_PLOTS/STGMIMS_vs_CGPS.jpg')

## Read in Dwingeloo data

In [ ]:
l_list = []
b_list = []
ra_list = []
dec_list = []
PI_list = []
gal_theta_list = []
eq_theta_list = []
with open('/srv/data/dwingeloo_data/original/out1411') as f:
    for line in f:
        line = f.readline()
        l_list.append(float(line[0:7]))
        b_list.append(float(line[7:13]))
        ra_list.append(float(line[13:19]))
        dec_list.append(float(line[19:25]))
        PI_list.append(float(line[25:31]))
        gal_theta_list.append(float(line[31:37]))
        eq_theta_list.append(float(line[37:43]))
f.close()

l_arr0 = np.asarray(l_list)
b_arr0 = np.asarray(b_list)
b_arr  = b_arr0.copy()
l_arr  = l_arr0.copy()
b_arr[b_arr0>270] = b_arr[b_arr0>270]-360.
l_arr[l_arr0>180] = l_arr[l_arr0>180]-360.

Qpts = np.asarray(PI_list)*np.cos((2*np.asarray(gal_theta_list))*np.pi/180)
Upts = np.asarray(PI_list)*np.sin((2*np.asarray(gal_theta_list))*np.pi/180)

## Quick plot to check

In [ ]:
fig = plt.figure(figsize=(16,12))

ax1,ax2 = [fig.add_subplot(211),fig.add_subplot(212)]
ax1.scatter(l_arr,b_arr,c=Qpts,cmap='RdBu_r',vmin=-0.5,vmax=0.5)
ax2.scatter(l_arr,b_arr,c=Upts,cmap='RdBu_r',vmin=-0.5,vmax=0.5)
for ax in [ax1,ax2]:
    ax.set_xlim(180,-180)
    ax.set_aspect('equal')

## Average GMIMS and DRAO 26-m points within Dwingeloo beam

In [ ]:
wcs1 = WCS(hdr_26)
wcs2 = WCS(hdr_G)

print(Q_26.shape)
print(repr(wcs1))
print(Q_G.shape)
print(repr(wcs2))

c1 = SkyCoord.from_pixel(np.arange(Q_26.shape[1])[:,np.newaxis],
                         np.arange(Q_26.shape[0])[np.newaxis,:],
                         wcs1.celestial)
c2= SkyCoord.from_pixel(np.arange(Q_G.shape[1])[:,np.newaxis],
                        np.arange(Q_G.shape[0])[np.newaxis,:],
                        wcs2.celestial)

radius = 0.3 #* u.deg
Q_26_avg = []
U_26_avg = []
Q_G_avg = []
U_G_avg = []
for i in np.arange(len(l_arr)):

    sep1 = angular_separation(np.radians(l_arr[i]), np.radians(b_arr[i]), 
                              c1.galactic.l.rad, c1.galactic.b.rad)*180/np.pi
    idx1 = np.where(sep1 <= radius)
    sep2 = angular_separation(np.radians(l_arr[i]), np.radians(b_arr[i]), 
                              c2.galactic.l.rad, c2.galactic.b.rad)*180/np.pi
    idx2 = np.where(sep2 <= radius)
    print(i)
    #print(l_arr[i],b_arr[i])
    #print(c.galactic.l.deg[idx])
    #print(c.galactic.b.deg[idx])
    #print('')
    Q_26_avg.append(np.nanmean(Q_26[idx1[1], idx1[0]]))
    U_26_avg.append(np.nanmean(U_26[idx1[1], idx1[0]])) 
    Q_G_avg.append(np.nanmean(Q_G[idx2[1], idx2[0]]))
    U_G_avg.append(np.nanmean(U_G[idx2[1], idx2[0]]))    

In [ ]:
fig = plt.figure(figsize=(16,24))

ax1,ax2,ax3,ax4,ax5,ax6 = [fig.add_subplot(621),fig.add_subplot(622),
                           fig.add_subplot(623),fig.add_subplot(624),
                           fig.add_subplot(625),fig.add_subplot(626)]

s = 10

ax1.scatter(l_arr,b_arr,c=Q_26_avg,cmap='RdBu_r',vmin=-500,vmax=500,s=10)
ax2.scatter(l_arr,b_arr,c=U_26_avg,cmap='RdBu_r',vmin=-500,vmax=500,s=10)
ax3.scatter(l_arr,b_arr,c=Qpts,cmap='RdBu_r',vmin=-0.5,vmax=0.5,s=10)
ax4.scatter(l_arr,b_arr,c=Upts,cmap='RdBu_r',vmin=-0.5,vmax=0.5,s=10)
ax5.scatter(l_arr,b_arr,c=Q_G_avg,cmap='RdBu_r',vmin=-0.5,vmax=0.5,s=10)
ax6.scatter(l_arr,b_arr,c=U_G_avg,cmap='RdBu_r',vmin=-0.5,vmax=0.5,s=10)

ax1.set_title('W06 Stokes Q')
ax2.set_title('W06 Stokes U')
ax3.set_title('Dwingeloo Stokes Q')
ax4.set_title('Dwingeloo Stokes U')
ax5.set_title('HBN Stokes Q')
ax6.set_title('HBN Stokes U')

for ax in [ax1,ax2,ax3,ax4,ax5,ax6]:
    ax.set_xlim(180,-180)
    ax.set_aspect('equal')

In [ ]:
def dwingeloo_compare(QDW,UDW,Q26,U26,QG,UG,l,b,
                      llim=[192,52], blim=[-7,10],filename='test',
                      *args,**kwargs):
    
    fs = 15
    s = 10
    
    ij = np.where((l<=llim[0]) & (l>=llim[1]) & (b<=blim[1]) & (b>=blim[0]))
    #print(l[ij])
    #print(b[ij])
    
    QDW_plt  = np.array(QDW)[ij]
    UDW_plt  = np.array(UDW)[ij]
    PIDW_plt = np.sqrt(QDW_plt**2+UDW_plt**2)
    PADW_plt = 0.5*np.arctan2(UDW_plt,QDW_plt)*180/np.pi
    
    Q26_plt  = np.array(Q26)[ij]/1e3
    U26_plt  = np.array(U26)[ij]/1e3
    PI26_plt = np.sqrt(Q26_plt**2+U26_plt**2)
    PA26_plt = 0.5*np.arctan2(U26_plt,Q26_plt)*180/np.pi
    
    QG_plt  = np.array(QG)[ij]
    UG_plt  = np.array(UG)[ij]
    PIG_plt = np.sqrt(QG_plt**2+UG_plt**2)
    PAG_plt = 0.5*np.arctan2(UG_plt,QG_plt)*180/np.pi
    
    # Set up axes:
    fig = plt.figure(figsize=(24,12))   
    plt.subplots_adjust(wspace=0.3,hspace=0.3)  

    axes = [fig.add_subplot(241),fig.add_subplot(242),fig.add_subplot(243),fig.add_subplot(244),
            fig.add_subplot(245),fig.add_subplot(246),fig.add_subplot(247),fig.add_subplot(248)]
    
    # DRAO 26-m:
    axes[0].scatter(QDW_plt, Q26_plt,  s=s,color='b')
    axes[1].scatter(UDW_plt, U26_plt,  s=s,color='r')
    axes[2].scatter(PIDW_plt,PI26_plt, s=s,color='purple')
    axes[3].scatter(PADW_plt,PA26_plt, s=s,color='g')
    
    axes[0].set_xlabel('Dwingeloo Q (K)',fontsize=fs)
    axes[1].set_xlabel('Dwingeloo U (K)',fontsize=fs)
    axes[2].set_xlabel('Dwingeloo PI (K)',fontsize=fs)
    axes[3].set_xlabel('Dwingeloo PA (deg)',fontsize=fs)
    
    axes[0].set_ylabel('DRAO 26-m Q (K)',fontsize=fs)
    axes[1].set_ylabel('DRAO 26-m U (K)',fontsize=fs)
    axes[2].set_ylabel('DRAO 26-m PI (K)',fontsize=fs)
    axes[3].set_ylabel('DRAO 26-m PA (deg)',fontsize=fs)
    
    axes[0].set_title('r='+str(round(pearsonr(QDW_plt,Q26_plt)[0],2)),fontsize=fs)
    axes[1].set_title('r='+str(round(pearsonr(UDW_plt,U26_plt)[0],2)),fontsize=fs)
    axes[2].set_title('r='+str(round(pearsonr(PIDW_plt,PI26_plt)[0],2)),fontsize=fs)
    #axes[3].set_title('r='+str(round(pearsonr(PADW_plt,PA26_plt)[0],2)),fontsize=fs)
        
    # GMIMS:
    axes[4].scatter(QDW_plt, QG_plt,  s=s,color='b')
    axes[5].scatter(UDW_plt, UG_plt,  s=s,color='r')
    axes[6].scatter(PIDW_plt,PIG_plt, s=s,color='purple')
    axes[7].scatter(PADW_plt,PAG_plt, s=s,color='g')
    
    axes[4].set_xlabel('Dwingeloo Q (K)',fontsize=fs)
    axes[5].set_xlabel('Dwingeloo U (K)',fontsize=fs)
    axes[6].set_xlabel('Dwingeloo PI (K)',fontsize=fs)
    axes[7].set_xlabel('Dwingeloo PA (deg)',fontsize=fs)
    
    axes[4].set_ylabel('GMIMS Q (K)',fontsize=fs)
    axes[5].set_ylabel('GMIMS U (K)',fontsize=fs)
    axes[6].set_ylabel('GMIMS PI (K)',fontsize=fs)
    axes[7].set_ylabel('GMIMS PA (deg)',fontsize=fs)
    
    axes[4].set_title('r='+str(round(pearsonr(QDW_plt[np.isfinite(QG_plt)],
                                              QG_plt[np.isfinite(QG_plt)])[0],2)),fontsize=fs)
    axes[5].set_title('r='+str(round(pearsonr(UDW_plt[np.isfinite(QG_plt)],
                                              UG_plt[np.isfinite(QG_plt)])[0],2)),fontsize=fs)
    axes[6].set_title('r='+str(round(pearsonr(PIDW_plt[np.isfinite(QG_plt)],
                                              PIG_plt[np.isfinite(QG_plt)])[0],2)),fontsize=fs)
    #axes[7].set_title('r='+str(round(pearsonr(PADW_plt[np.isfinite(QG_plt)],
    #                                          PAG_plt[np.isfinite(QG_plt)])[0],2)),fontsize=fs)
    
    for ax in axes:
        ax.set_aspect('equal')
        ax.set_xlim(-0.8,0.8)
        ax.set_ylim(-0.8,0.8)
        ax.set_xticks([-0.8,-0.4,0,0.4,0.8])
        ax.set_yticks([-0.8,-0.4,0,0.4,0.8])
        ax.grid()
        ax.tick_params(labelsize=fs-2)
        ax.plot([-1,1],[-1,1],color='k',linewidth=0.5)
    axes[2].set_xlim(0,0.8)
    axes[6].set_xlim(0,0.8)
    axes[2].set_xticks([0,0.2,0.4,0.6,0.8])
    axes[6].set_yticks([0,0.2,0.4,0.6,0.8])
    
    axes[3].set_xlim(-90,90)
    axes[7].set_xlim(-90,90)
    
    axes[2].set_ylim(0,0.8)
    axes[6].set_ylim(0,0.8)
    axes[3].set_ylim(-90,90)
    axes[7].set_ylim(-90,90)
    
    axes[3].set_xticks([-90,-45,0,45,90])
    axes[7].set_xticks([-90,-45,0,45,90])
    axes[3].set_yticks([-90,-45,0,45,90])
    axes[7].set_yticks([-90,-45,0,45,90])
    
    #plt.savefig('/home/aordog/CGPS_GMIMS_PLOTS/'+filename+'.jpg')

    return

In [ ]:
dwingeloo_compare(Qpts,Upts,Q_26_avg,U_26_avg,Q_G_avg,U_G_avg,l_arr,b_arr,
                  llim=[160,120], blim=[-3.5,5.5],filename='Fan_region')

In [ ]:
dwingeloo_compare(Qpts,Upts,Q_26_avg,U_26_avg,Q_G_avg,U_G_avg,l_arr,b_arr,
                  llim=[180,60], blim=[-3,5],filename='Dwingeloo_cgps')

In [ ]:
def dwingeloo_compare_v2(QDW,UDW,Q26,U26,QG,UG,l,b,
                      llim=[192,52], blim=[-7,10],
                      llim_sub=[160,120], blim_sub=[-7,10],
                      filename='test',
                      *args,**kwargs):
    
    fs = 18
    s = 20
    
    ij  = np.where((l<=llim[0]) & (l>=llim[1]) & (b<=blim[1]) & (b>=blim[0]))
    ij2 = np.where((l<=llim_sub[0]) & (l>=llim_sub[1]) & (b<=blim_sub[1]) & (b>=blim_sub[0]))
    
    QDW_plt  = np.array(QDW)[ij]
    UDW_plt  = np.array(UDW)[ij]
    PIDW_plt = np.sqrt(QDW_plt**2+UDW_plt**2)
    PADW_plt = 0.5*np.arctan2(UDW_plt,QDW_plt)*180/np.pi
    
    Q26_plt  = np.array(Q26)[ij]/1e3
    U26_plt  = np.array(U26)[ij]/1e3
    PI26_plt = np.sqrt(Q26_plt**2+U26_plt**2)
    PA26_plt = 0.5*np.arctan2(U26_plt,Q26_plt)*180/np.pi
    
    QG_plt  = np.array(QG)[ij]
    UG_plt  = np.array(UG)[ij]
    PIG_plt = np.sqrt(QG_plt**2+UG_plt**2)
    PAG_plt = 0.5*np.arctan2(UG_plt,QG_plt)*180/np.pi
    
    # Set up axes:
    fig, axs = plt.subplots(3,4,figsize=(18,12))
    plt.subplots_adjust(wspace=0.4,hspace=0.3,bottom=0.06,left=0.06,top=0.99,right=0.99)  

    # DRAO 26-m:
    axs[0,0].scatter(QDW_plt, Q26_plt,  s=s,color='gray')
    axs[0,1].scatter(UDW_plt, U26_plt,  s=s,color='gray')
    axs[0,2].scatter(PIDW_plt,PI26_plt, s=s,color='gray')
    axs[0,3].scatter(PADW_plt,PA26_plt, s=s,color='gray')
    
    axs[0,0].set_xlabel('Dwingeloo Q (K)',fontsize=fs)
    axs[0,1].set_xlabel('Dwingeloo U (K)',fontsize=fs)
    axs[0,2].set_xlabel('Dwingeloo PI (K)',fontsize=fs)
    axs[0,3].set_xlabel('Dwingeloo PA (deg)',fontsize=fs)
    
    axs[0,0].set_ylabel('W06 Q (K)',fontsize=fs)
    axs[0,1].set_ylabel('W06 U (K)',fontsize=fs)
    axs[0,2].set_ylabel('W06 PI (K)',fontsize=fs)
    axs[0,3].set_ylabel('W06 PA (deg)',fontsize=fs)
    
    axs[0,0].text(-0.6,0.6,'r='+str(round(pearsonr(QDW_plt,Q26_plt)[0],2)),fontsize=fs)
    axs[0,1].text(-0.6,0.6,'r='+str(round(pearsonr(UDW_plt,U26_plt)[0],2)),fontsize=fs)
    axs[0,2].text( 0.1,0.7,'r='+str(round(pearsonr(PIDW_plt,PI26_plt)[0],2)),fontsize=fs)
    axs[0,3].text(-67.5,67.5,'r='+str(round(pearsonr(PADW_plt,PA26_plt)[0],2)),fontsize=fs)
        
    # GMIMS:
    axs[1,0].scatter(QDW_plt, QG_plt,  s=s,color='gray')
    axs[1,1].scatter(UDW_plt, UG_plt,  s=s,color='gray')
    axs[1,2].scatter(PIDW_plt,PIG_plt, s=s,color='gray')
    axs[1,3].scatter(PADW_plt,PAG_plt, s=s,color='gray')
    
    axs[1,0].set_xlabel('Dwingeloo Q (K)',fontsize=fs)
    axs[1,1].set_xlabel('Dwingeloo U (K)',fontsize=fs)
    axs[1,2].set_xlabel('Dwingeloo PI (K)',fontsize=fs)
    axs[1,3].set_xlabel('Dwingeloo PA (deg)',fontsize=fs)
    
    axs[1,0].set_ylabel('GMIMS-HBN Q (K)',fontsize=fs)
    axs[1,1].set_ylabel('GMIMS-HBN U (K)',fontsize=fs)
    axs[1,2].set_ylabel('GMIMS-HBN PI (K)',fontsize=fs)
    axs[1,3].set_ylabel('GMIMS-HBN PA (deg)',fontsize=fs)

    axs[1,0].text(-0.6,0.6,'r='+str(round(pearsonr(QDW_plt,QG_plt)[0],2)),fontsize=fs)
    axs[1,1].text(-0.6,0.6,'r='+str(round(pearsonr(UDW_plt,UG_plt)[0],2)),fontsize=fs)
    axs[1,2].text( 0.1,0.7,'r='+str(round(pearsonr(PIDW_plt,PIG_plt)[0],2)),fontsize=fs)
    axs[1,3].text(-67.5,67.5,'r='+str(round(pearsonr(PADW_plt,PAG_plt)[0],2)),fontsize=fs)

    
    # GMIMS vs W06:
    axs[2,0].scatter(Q26_plt, QG_plt,  s=s,color='gray')
    axs[2,1].scatter(U26_plt, UG_plt,  s=s,color='gray')
    axs[2,2].scatter(PI26_plt,PIG_plt, s=s,color='gray')
    axs[2,3].scatter(PA26_plt,PAG_plt, s=s,color='gray')
    
    axs[2,0].set_xlabel('W06 Q (K)',fontsize=fs)
    axs[2,1].set_xlabel('W06 U (K)',fontsize=fs)
    axs[2,2].set_xlabel('W06 PI (K)',fontsize=fs)
    axs[2,3].set_xlabel('W06 PA (deg)',fontsize=fs)
    
    axs[2,0].set_ylabel('GMIMS-HBN Q (K)',fontsize=fs)
    axs[2,1].set_ylabel('GMIMS-HBN U (K)',fontsize=fs)
    axs[2,2].set_ylabel('GMIMS-HBN PI (K)',fontsize=fs)
    axs[2,3].set_ylabel('GMIMS-HBN PA (deg)',fontsize=fs)

    axs[2,0].text(-0.6,0.6,'r='+str(round(pearsonr(Q26_plt,QG_plt)[0],2)),fontsize=fs)
    axs[2,1].text(-0.6,0.6,'r='+str(round(pearsonr(U26_plt,UG_plt)[0],2)),fontsize=fs)
    axs[2,2].text( 0.1,0.7,'r='+str(round(pearsonr(PI26_plt,PIG_plt)[0],2)),fontsize=fs)
    axs[2,3].text(-67.5,67.5,'r='+str(round(pearsonr(PA26_plt,PAG_plt)[0],2)),fontsize=fs)


    QDW_plt  = np.array(QDW)[ij2]
    UDW_plt  = np.array(UDW)[ij2]
    PIDW_plt = np.sqrt(QDW_plt**2+UDW_plt**2)
    PADW_plt = 0.5*np.arctan2(UDW_plt,QDW_plt)*180/np.pi
    
    Q26_plt  = np.array(Q26)[ij2]/1e3
    U26_plt  = np.array(U26)[ij2]/1e3
    PI26_plt = np.sqrt(Q26_plt**2+U26_plt**2)
    PA26_plt = 0.5*np.arctan2(U26_plt,Q26_plt)*180/np.pi
    
    QG_plt  = np.array(QG)[ij2]
    UG_plt  = np.array(UG)[ij2]
    PIG_plt = np.sqrt(QG_plt**2+UG_plt**2)
    PAG_plt = 0.5*np.arctan2(UG_plt,QG_plt)*180/np.pi

    axs[0,0].scatter(QDW_plt, Q26_plt,  s=s,color='k')
    axs[0,1].scatter(UDW_plt, U26_plt,  s=s,color='k')
    axs[0,2].scatter(PIDW_plt,PI26_plt, s=s,color='k')
    axs[0,3].scatter(PADW_plt,PA26_plt, s=s,color='k')

    axs[1,0].scatter(QDW_plt, QG_plt,  s=s,color='k')
    axs[1,1].scatter(UDW_plt, UG_plt,  s=s,color='k')
    axs[1,2].scatter(PIDW_plt,PIG_plt, s=s,color='k')
    axs[1,3].scatter(PADW_plt,PAG_plt, s=s,color='k')

    axs[2,0].scatter(Q26_plt, QG_plt,  s=s,color='k')
    axs[2,1].scatter(U26_plt, UG_plt,  s=s,color='k')
    axs[2,2].scatter(PI26_plt,PIG_plt, s=s,color='k')
    axs[2,3].scatter(PA26_plt,PAG_plt, s=s,color='k')

    for j in range(0,3):
        for i in range(0,4):
            axs[j,i].set_aspect('equal')
            axs[j,i].grid()
            axs[j,i].tick_params(labelsize=fs-2)
            axs[j,i].plot([-1,1],[-1,1],color='k',linewidth=0.5)
            if i == 2:
                axs[j,i].set_xlim(0,0.8)
                axs[j,i].set_ylim(0,0.8)
                axs[j,i].set_xticks([0,0.2,0.4,0.6,0.8])
                axs[j,i].set_yticks([0,0.2,0.4,0.6,0.8])
            if i == 3:
                axs[j,i].set_xlim(-90,90)
                axs[j,i].set_ylim(-90,90)
                axs[j,i].set_xticks([-90,-45,0,45,90])
                axs[j,i].set_yticks([-90,-45,0,45,90])
                axs[j,i].plot([-90,90],[-90,90],color='k',linewidth=0.5)
        for i in range(0,2):
            axs[j,i].set_xlim(-0.8,0.8)
            axs[j,i].set_ylim(-0.8,0.8)
            axs[j,i].set_xticks([-0.8,-0.4,0,0.4,0.8])
            axs[j,i].set_yticks([-0.8,-0.4,0,0.4,0.8])
    
    plt.savefig('../plots/'+filename+'.png')

    return

In [ ]:
dwingeloo_compare_v2(Qpts,Upts,Q_26_avg,U_26_avg,Q_G_avg,U_G_avg,l_arr,b_arr,
                  llim=[180,60], blim=[-3.5,5.5],
                  llim_sub=[160,120], blim_sub=[-3.5,5.5],
                  filename='Dwingeloo_compare_cgps_new')

In [ ]:
plt.hist(PI_26.flatten()/1000,bins=200,range=(0,1));
plt.hist(PI_G.flatten(),bins=200,range=(0,1),alpha=0.5);
plt.xlim(0,1)

In [ ]:
print(PI_26.shape)
print(PI_G.shape)
plt.imshow((PI_26[:,0:1440]/1000-PI_G)/PI_G,origin='lower',cmap='RdBu_r',vmin=-1,vmax=1)

In [ ]:
plt.imshow((PI_26[:,1:1441]/1000-PI_G)/PI_G,origin='lower',cmap='RdBu_r',vmin=-1,vmax=1)

In [ ]:
def make_single_dish_maps(data,hdrs,vmin=[0,0,0],vmax=[1,1,1],cmap=['viridis','viridis','viridis'],
                         llim = [192,52], blim = [-7,10],filename='test',
                         *args,**kwargs):
    
    c = SkyCoord(llim, blim, frame=Galactic, unit="deg")
    fs = 16
    
    # Set up axes:
    #fig = plt.figure(figsize=(8,11))
    fig = plt.figure(figsize=(16,22))  
    plt.subplots_adjust(hspace=0.1,left=0.12, right=0.88, top=0.95, bottom=0.1)  
    ax1  = fig.add_subplot(311, projection=WCS(hdrs[0]).celestial)
    ax2  = fig.add_subplot(312, projection=WCS(hdrs[1]).celestial)
    ax3  = fig.add_subplot(313, projection=WCS(hdrs[2]).celestial) 
    axes = [ax1,ax2,ax3]
    
    # The maps:
    ims   = []
    for i in range(0,3):
        ims.append(axes[i].imshow(data[i],origin='lower',vmin=vmin[i], vmax=vmax[i],cmap=cmap[i]))
        axes[i].set_xlim(WCS(hdrs[i]).world_to_pixel(c)[0])
        axes[i].set_ylim(WCS(hdrs[i]).world_to_pixel(c)[1])
    axes[0].set_title(r'DRAO 26-m PI (K)', fontsize=fs)
    axes[1].set_title(r'GMIMS PI (K)', fontsize=fs)
    axes[2].set_title(r'DRAO 26-m PI - GMIMS PI', fontsize=fs)
    
    # The colorbars
    cbar1 = fig.colorbar(ims[0],ax=axes[0],orientation='vertical',
                         fraction=0.02,pad=0.01,aspect=20)
    cbar1.set_ticks([0,0.1,0.2,0.3,0.4,0.5])
    cbar1.set_label(r'PI (K)', fontsize=fs)
    
    cbar2 = fig.colorbar(ims[1],ax=axes[1],orientation='vertical',
                         fraction=0.02,pad=0.01,aspect=20)
    cbar2.set_ticks([0,0.1,0.2,0.3,0.4,0.5])
    cbar2.set_label(r'PI (K)', fontsize=fs)
    
    cbar3 = fig.colorbar(ims[2],ax=axes[2],orientation='vertical',
                         fraction=0.02,pad=0.01,aspect=20)
    cbar3.set_ticks([-0.3,-0.2,-0.1,0,0.1,0.2,0.3])
    cbar3.set_label('difference (K)', fontsize=fs)
    
    for cbar in [cbar1,cbar2,cbar3]:
        cbar.ax.tick_params(axis='y', which='both', width=2, length=6)
        cbar.ax.tick_params(labelsize=fs)
        cbar.outline.set_linewidth(2)
    
    for ax in [ax1,ax2,ax3]:
        ax.tick_params(axis='both', labelsize=fs)
        ax.set_ylabel('  ',fontsize=fs)
        ax.set_xlabel('  ',fontsize=fs)
        ax.tick_params(axis='both', which='both', width=2, length=6)
        for spine in ax.spines.values():
            spine.set_visible(True)
            spine.set_linewidth(2)
    ax3.set_ylabel('Galactic Latitude',fontsize=fs)
    ax3.set_xlabel('Galactic Longitude',fontsize=fs)
    ax1.set_ylabel('Galactic Latitude',fontsize=fs)
    ax2.set_ylabel('Galactic Latitude',fontsize=fs)

    
    plt.savefig('/home/aordog/CGPS_GMIMS_PLOTS/'+filename+'.jpg')

    return

In [ ]:
l1 = -179
l2 = 180

make_single_dish_maps([PI_26[:,0:1440]/1000, PI_G, (PI_26[:,0:1440]/1000-PI_G)/PI_G],
                     [hdr_26,hdr_G,hdr_G],vmin = [0,0,-1],
                     vmax=[0.6,0.6,1],cmap=['gist_heat_r','gist_heat_r','RdBu_r'],
                     llim = [l2,l1], blim = [-90,90],
                     filename = 'SA_PI_compare_'+str(l1)+'_'+str(l2))

In [ ]:
l1 = -179
l2 = 180

make_single_dish_maps([PI_26[:,0:1440]/1000, PI_G, PI_26[:,0:1440]/1000-PI_G],
                     [hdr_26,hdr_G,hdr_G],vmin = [0,0,-0.3],
                     vmax=[0.6,0.6,0.3],cmap=['gist_heat_r','gist_heat_r','RdBu_r'],
                     llim = [l2,l1], blim = [-90,90],
                     filename = 'SA_PI_compare_diff')